# 파이썬 머신러닝 완벽 가이드 4장

LightGBM, HyperOpt, 스태킹 앙상블

## 실습 환경 설정

과제 조건에 따라 scikit-learn 1.2.2와 XGBoost 1.7.6을 설치한다. 설치 후에는 **런타임 → 세션 다시 시작**을 한 번 실행한 뒤 아래 셀부터 진행한다.

In [ ]:
!pip install scikit-learn==1.2.2
!pip install xgboost==1.7.6
!pip install hyperopt

In [ ]:
import sklearn
import xgboost
import lightgbm

print(sklearn.__version__, xgboost.__version__, lightgbm.__version__)

## 07. LightGBM

### 개념 정리

LightGBM은 XGBoost와 함께 가장 널리 쓰이는 부스팅 알고리즘이다. 예측 성능은 XGBoost와 비슷하지만 **학습 시간이 짧고 메모리 사용량이 적다**.

**리프 중심 트리 분할(Leaf Wise)**
- 대부분의 트리 알고리즘은 트리의 균형을 맞추며 깊이를 늘리는 **균형 트리 분할(Level Wise)** 방식을 쓴다. 균형 트리는 과적합에 강하지만 균형을 맞추는 데 시간이 든다.
- LightGBM은 균형을 맞추지 않고, **손실을 가장 많이 줄일 수 있는 리프 노드를 계속 분할**한다. 그래서 트리가 깊고 비대칭적으로 자란다.
- 학습을 반복할수록 균형 트리 방식보다 예측 오류를 더 줄일 수 있다는 것이 LightGBM의 설계 아이디어이다.

**XGBoost 대비 장점**
- 학습과 예측 속도가 더 빠르다.
- 메모리를 더 적게 사용한다.
- 카테고리형 피처를 원-핫 인코딩 없이 자동으로 변환하고 최적으로 분할한다.
- 병렬 처리와 GPU 학습을 지원한다.

**단점**: 데이터가 적으면(공식 문서 기준 약 10,000건 이하) 과적합이 일어나기 쉽다.

**패키지 구성**: `lightgbm` 패키지 안에 파이썬 래퍼와 사이킷런 래퍼(`LGBMClassifier`, `LGBMRegressor`)가 함께 들어 있다. 사이킷런 래퍼는 `fit()`/`predict()`와 사이킷런 유틸리티를 그대로 쓸 수 있다.

In [ ]:
import lightgbm
from lightgbm import LGBMClassifier

### 주요 하이퍼 파라미터

| 파라미터(파이썬 래퍼) | 사이킷런 래퍼 | 기본값 | 의미 |
|---|---|---|---|
| `num_iterations` | `n_estimators` | 100 | 반복할 트리 개수. 너무 크면 과적합이 생길 수 있다 |
| `learning_rate` | `learning_rate` | 0.1 | 부스팅 단계마다 적용하는 학습률 |
| `max_depth` | `max_depth` | -1 | 트리 최대 깊이. 0보다 작으면 제한이 없다 |
| `min_data_in_leaf` | `min_child_samples` | 20 | 리프 노드가 되기 위한 최소 데이터 수. 과적합을 제어한다 |
| `num_leaves` | `num_leaves` | 31 | 트리 하나가 가질 수 있는 최대 리프 수 |
| `boosting` | `boosting_type` | `gbdt` | 트리 생성 방식(`gbdt`: 그래디언트 부스팅, `rf`: 랜덤 포레스트) |
| `bagging_fraction` | `subsample` | 1.0 | 데이터 샘플링 비율 |
| `feature_fraction` | `colsample_bytree` | 1.0 | 트리마다 무작위로 고르는 피처 비율 |
| `lambda_l2` | `reg_lambda` | 0.0 | L2 규제 |
| `lambda_l1` | `reg_alpha` | 0.0 | L1 규제 |
| `objective` | `objective` | - | 손실 함수(회귀, 이진 분류, 다중 분류에 따라 지정) |

- 사이킷런 래퍼 LightGBM은 사이킷런 래퍼 XGBoost와 파라미터 이름을 맞췄기 때문에 대부분 이름이 같다.
- Leaf Wise 방식이라 트리가 깊어지므로, 트리 특성에 맞게 파라미터를 설정해야 한다.

### 튜닝 방향

1. **`num_leaves`를 중심으로** `min_child_samples`, `max_depth`를 함께 조정해 모델의 복잡도를 줄인다.
   - `num_leaves`를 늘리면 정확도는 오를 수 있지만 트리가 복잡해져 과적합 위험이 커진다.
   - `min_child_samples`를 크게 하면 트리가 지나치게 깊어지는 것을 막는다.
   - `max_depth`는 깊이를 직접 제한한다.
2. `learning_rate`를 낮추고 `n_estimators`를 늘린다. 단, `n_estimators`가 너무 크면 과적합이 생길 수 있다.
3. `reg_lambda`, `reg_alpha`로 규제하거나 `colsample_bytree`, `subsample`로 학습에 쓰는 피처와 데이터 수를 줄인다.

### LightGBM 적용: 위스콘신 유방암 예측

XGBoost 실습과 같은 방식으로 학습 80%, 테스트 20%로 나누고, 학습 데이터를 다시 최종 학습 90%, 검증 10%로 나눈다. 최신 LightGBM에서는 조기 중단을 `fit()`의 `callbacks` 인자로 지정한다.

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()
cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target'] = dataset.target

X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# 학습 80% / 테스트 20%
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, test_size=0.2, random_state=156)

# 학습 데이터를 다시 최종 학습 90% / 검증 10%로 분할
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=156)

lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM도 XGBoost와 동일하게 조기 중단 수행 가능
evals=[(X_tr, y_tr),(X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=10)], eval_metric="logloss", eval_set=evals)
preds=lgbm_wrapper.predict(X_test)
pred_proba=lgbm_wrapper.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

def get_clf_eval(y_test, pred, pred_proba):
    confusion=confusion_matrix(y_test, pred)
    accuracy=accuracy_score(y_test, pred)
    precision=precision_score(y_test, pred)
    recall=recall_score(y_test, pred)
    f1=f1_score(y_test, pred)
    #ROC-AUC 추가
    roc_auc=roc_auc_score(y_test, pred_proba)
    print('오차행렬')
    print(confusion)
    #ROC-AUC print 추가
    print('정확도:{0:.4f}, 정밀도:{1:.4f}, 재현율{2:.4f}, F1:{3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall,f1,roc_auc))

In [ ]:
get_clf_eval(y_test, preds, pred_proba)

LightGBM도 `plot_importance()`로 피처 중요도를 바로 시각화할 수 있으며, 사이킷런 래퍼 객체를 그대로 넣어도 된다.

In [ ]:
from lightgbm import plot_importance
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(lgbm_wrapper, ax=ax)

## 08. 베이지안 최적화 기반의 HyperOpt를 이용한 하이퍼 파라미터 튜닝

### Grid Search의 한계

Grid Search는 모든 파라미터 조합을 하나씩 학습하고 평가한다. 예를 들어 6개 파라미터가 각각 5, 4, 5, 5, 4, 3개의 후보를 가지면 5×4×5×5×4×3 = 6,000번 학습해야 한다. XGBoost와 LightGBM은 파라미터가 많아서, 대용량 데이터에 Grid Search를 적용하면 시간이 지나치게 오래 걸린다.

### 베이지안 최적화

함수 식을 모르는 **블랙박스 함수**에서, 반환값을 최대 또는 최소로 만드는 입력값을 **적은 시도로** 찾는 방법이다. 새로운 관측값이 들어올 때마다 사후 확률을 갱신하는 베이지안 확률의 아이디어를 이용한다. 하이퍼 파라미터 튜닝에서 입력값은 하이퍼 파라미터, 반환값은 모델 성능이다.

| 구성 요소 | 역할 |
|---|---|
| 대체 모델(Surrogate Model) | 지금까지 관측한 결과로 목표 함수의 모양을 추정한다 |
| 획득 함수(Acquisition Function) | 추정된 함수를 바탕으로, 더 좋은 값을 낼 가능성이 높은 다음 입력값을 고른다 |

**진행 단계**
1. 처음에는 하이퍼 파라미터를 무작위로 몇 개 골라 성능을 관측한다.
2. 대체 모델이 관측값으로 최적 함수와 그 불확실성(신뢰 구간)을 추정한다.
3. 획득 함수가 지금까지의 최고값보다 좋아질 가능성이 큰 지점을 다음 관측 대상으로 고른다.
4. 그 값으로 관측한 결과를 반영해 대체 모델을 갱신한다.
5. 3~4단계를 반복할수록 추정이 정확해진다.

대체 모델로는 보통 가우시안 프로세스를 쓰지만, HyperOpt는 **TPE(Tree-structured Parzen Estimator)**를 쓴다.

### HyperOpt 사용 흐름

1. **검색 공간 설정**: 딕셔너리의 키는 변수 이름, 값은 `hp` 모듈 함수로 지정한 검색 범위이다.
2. **목적 함수 정의**: 검색 공간 딕셔너리를 입력받아 값(또는 `{'loss': 값, 'status': STATUS_OK}`)을 반환한다.
3. **`fmin()`으로 최적값 탐색**: HyperOpt는 반환값이 **최소**가 되는 입력값을 찾는다.

| 검색 공간 함수 | 의미 |
|---|---|
| `hp.quniform(label, low, high, q)` | low부터 high까지 q 간격의 값 |
| `hp.uniform(label, low, high)` | low부터 high까지의 연속 균등 분포 |
| `hp.randint(label, upper)` | 0부터 upper까지의 무작위 정수 |
| `hp.loguniform(label, low, high)` | `exp(uniform(low, high))` 값. 로그를 취하면 균등 분포가 된다 |
| `hp.choice(label, options)` | 리스트나 튜플로 준 후보 중 하나(문자열 가능) |

| `fmin()` 인자 | 의미 |
|---|---|
| `fn` | 목적 함수 |
| `space` | 검색 공간 딕셔너리 |
| `algo` | 최적화 알고리즘. 기본은 `tpe.suggest` |
| `max_evals` | 시도 횟수 |
| `trials` | 시도한 입력값과 결과를 저장할 `Trials` 객체 |
| `rstate` | 결과 재현을 위한 랜덤 시드. `np.random.default_rng(seed=0)` 형태로 넣는다 |

In [ ]:
from hyperopt import hp

# x는 -10~10, y는 -15~15 범위에서 1 간격으로 탐색
search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1)}

In [ ]:
from hyperopt import STATUS_OK

# 검색 공간 딕셔너리를 받아 x^2 - 20y를 반환하는 목적 함수
def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y
    return retval

In [ ]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

# 5번의 시도로 목적 함수가 최소가 되는 x, y 탐색
best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5,
               trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_01)

In [ ]:
trial_val = Trials()

# 시도 횟수를 20번으로 늘려 다시 탐색
best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20,
               trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_02)

최솟값은 x=0, y=15일 때이다. x는 21가지, y는 31가지 값을 가질 수 있으므로 모든 조합을 시도하면 최대 651번이 필요하지만, 베이지안 최적화는 20번만으로 정답에 가까운 값을 찾는다.

`Trials` 객체에는 시도 과정이 저장된다.
- `results`: 매 시도의 반환값. `{'loss': 반환값, 'status': 상태}` 딕셔너리의 리스트이다.
- `vals`: 매 시도의 입력값. `{'변수명': 입력값 리스트}` 형태이다.

In [ ]:
print(trial_val.results)

In [ ]:
print(trial_val.vals)

In [ ]:
import pandas as pd

# results의 loss 값과 vals의 입력값을 DataFrame으로 정리
losses = [loss_dict['loss'] for loss_dict in trial_val.results]

result_df = pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses': losses})
result_df

### HyperOpt로 XGBoost 하이퍼 파라미터 최적화

주의할 점이 두 가지 있다.
1. **형변환**: HyperOpt가 넘겨주는 값은 모두 실수형이다. `hp.quniform('max_depth', 5, 20, 1)`도 5, 6이 아니라 5.0, 6.0을 넘겨주므로, 정수형 파라미터에는 `int()`를 씌워야 한다.
2. **부호 변환**: `fmin()`은 반환값을 최소화한다. 정확도처럼 클수록 좋은 지표는 **-1을 곱해** 반환해야 한다. MAE, RMSE처럼 작을수록 좋은 지표는 그대로 반환한다.

목적 함수의 반환값은 3폴드 교차 검증 평균 정확도로 하고, 시간을 줄이기 위해 `n_estimators=100`으로 둔다. `cross_val_score()`에서는 조기 중단을 쓸 수 없다.

In [ ]:
# 학습 80% / 테스트 20%
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, test_size=0.2, random_state=156)

# 학습 데이터를 다시 학습 / 검증으로 분할
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=156)

In [ ]:
from hyperopt import hp

# max_depth: 5~20(1 간격), min_child_weight: 1~2(1 간격)
# learning_rate: 0.01~0.2, colsample_bytree: 0.5~1 (연속 균등 분포)
xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 20, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1)
                   }

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

def objective_func(search_space):
    # 정수형 파라미터는 int()로 변환
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)

    # 정확도가 높을수록 반환값이 작아지도록 -1을 곱함
    return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK}

In [ ]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50,
            trials=trial_val, rstate=np.random.default_rng(seed=9))
print('best:', best)

In [ ]:
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))

찾은 파라미터로 `n_estimators=400`, 조기 중단 50회를 적용해 다시 학습하고 테스트 세트로 평가한다.

In [ ]:
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5),
                            early_stopping_rounds=50, eval_metric='logloss'
                           )

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, eval_set=evals, verbose=True)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

get_clf_eval(y_test, preds, pred_proba)

위스콘신 데이터는 569건으로 작고, 이를 다시 학습·검증·테스트로 나눴기 때문에 `rstate` 값에 따라 결과가 불안정할 수 있다.

## 11. 스태킹 앙상블

### 개념 정리

**스태킹(Stacking)**은 여러 개별 모델의 **예측 결과를 다시 학습 데이터로 사용**해, 별도의 **메타 모델**이 최종 예측을 하는 방식이다.

- 필요한 모델: ① 여러 개의 **개별 기반 모델**, ② 기반 모델의 예측값을 학습하는 **최종 메타 모델**
- 데이터 흐름: M개 행을 가진 데이터에 모델 3개를 쓰면, 각 모델이 M×1 예측값을 내고 이를 옆으로 붙여 M×3 데이터를 만든다. 메타 모델은 이 M×3 데이터로 최종 예측을 한다.
- 실무에서는 많이 쓰이지 않지만, 캐글처럼 성능을 조금이라도 높여야 할 때 자주 쓰인다.
- 모델 2~3개만으로는 효과가 작고, 성능 향상이 보장되지도 않는다. 보통 **성능이 비슷한 모델 여러 개**를 결합한다.

### 기본 스태킹 모델

개별 모델은 KNN, 랜덤 포레스트, 결정 트리, 에이다부스트이고, 메타 모델은 로지스틱 회귀이다.

In [ ]:
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

cancer_data = load_breast_cancer()

X_data = cancer_data.data
y_label = cancer_data.target

X_train, X_test, y_train, y_test = train_test_split(X_data, y_label, test_size=0.2, random_state=0)

In [ ]:
# 개별 기반 모델
knn_clf = KNeighborsClassifier(n_neighbors=4)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=0)
dt_clf = DecisionTreeClassifier()
ada_clf = AdaBoostClassifier(n_estimators=100)

# 최종 메타 모델
lr_final = LogisticRegression()

In [ ]:
knn_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)
dt_clf.fit(X_train, y_train)
ada_clf.fit(X_train, y_train)

In [ ]:
knn_pred = knn_clf.predict(X_test)
rf_pred = rf_clf.predict(X_test)
dt_pred = dt_clf.predict(X_test)
ada_pred = ada_clf.predict(X_test)

print('KNN 정확도: {0:.4f}'.format(accuracy_score(y_test, knn_pred)))
print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy_score(y_test, rf_pred)))
print('결정 트리 정확도: {0:.4f}'.format(accuracy_score(y_test, dt_pred)))
print('에이다부스트 정확도: {0:.4f}'.format(accuracy_score(y_test, ada_pred)))

각 모델의 예측값(길이 114인 1차원 배열 4개)을 쌓으면 (4, 114)가 된다. 이를 전치해 (114, 4) 형태로 바꾸면, 각 모델의 예측값이 하나의 피처(열)가 된다.

In [ ]:
pred = np.array([knn_pred, rf_pred, dt_pred, ada_pred])
print(pred.shape)

# 행과 열을 바꿔 모델별 예측값을 열(피처)로 만듦
pred = np.transpose(pred)
print(pred.shape)

In [ ]:
lr_final.fit(pred, y_test)
final = lr_final.predict(pred)

print('최종 메타 모델의 예측 정확도: {0:.4f}'.format(accuracy_score(y_test, final)))

이 기본 방식은 메타 모델이 **테스트 데이터의 레이블로 학습**하기 때문에 과적합 위험이 있다. 이를 보완한 것이 CV 세트 기반 스태킹이다.

### CV 세트 기반의 스태킹

메타 모델용 학습·테스트 데이터를 **교차 검증으로** 만들어 과적합을 줄인다.

**스텝 1: 개별 모델마다 메타 모델용 데이터 만들기** (폴드 3개 예시)
1. 원본 학습 데이터를 3개 폴드로 나눈다. 2개 폴드로 개별 모델을 학습하고, 남은 1개 폴드(검증 폴드)를 예측해 저장한다.
2. 검증 폴드를 바꿔 가며 3번 반복하면 학습 데이터 전체에 대한 예측값이 모인다. 이것이 **메타 모델의 학습 데이터**가 된다.
3. 매 반복에서 학습된 모델로 **원본 테스트 데이터**도 예측한다. 3번의 예측값을 평균한 것이 **메타 모델의 테스트 데이터**가 된다.

**스텝 2: 메타 모델 학습과 예측**
1. 모든 개별 모델이 만든 학습 데이터를 옆으로 붙여 최종 학습 데이터를, 테스트 데이터를 붙여 최종 테스트 데이터를 만든다.
2. 메타 모델은 최종 학습 데이터와 **원본 학습 레이블**로 학습한다.
3. 최종 테스트 데이터를 예측하고 **원본 테스트 레이블**로 평가한다.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# 개별 기반 모델로 메타 모델용 학습 데이터와 테스트 데이터를 만드는 함수
def get_stacking_base_datasets(model, X_train_n, y_train_n, X_test_n, n_folds):
    kf = KFold(n_splits=n_folds, shuffle=False)

    train_fold_pred = np.zeros((X_train_n.shape[0], 1))
    test_pred = np.zeros((X_test_n.shape[0], n_folds))
    print(model.__class__.__name__, ' model 시작 ')

    for folder_counter, (train_index, valid_index) in enumerate(kf.split(X_train_n)):
        print('\t 폴드 세트: ', folder_counter, ' 시작 ')
        X_tr = X_train_n[train_index]
        y_tr = y_train_n[train_index]
        X_te = X_train_n[valid_index]

        # 학습 폴드로 학습한 뒤 검증 폴드를 예측해 메타 모델 학습 데이터로 저장
        model.fit(X_tr, y_tr)
        train_fold_pred[valid_index, :] = model.predict(X_te).reshape(-1, 1)

        # 같은 모델로 원본 테스트 데이터를 예측해 저장
        test_pred[:, folder_counter] = model.predict(X_test_n)

    # 폴드별 테스트 예측값의 평균을 메타 모델 테스트 데이터로 사용
    test_pred_mean = np.mean(test_pred, axis=1).reshape(-1, 1)

    return train_fold_pred, test_pred_mean

In [ ]:
knn_train, knn_test = get_stacking_base_datasets(knn_clf, X_train, y_train, X_test, 7)
rf_train, rf_test = get_stacking_base_datasets(rf_clf, X_train, y_train, X_test, 7)
dt_train, dt_test = get_stacking_base_datasets(dt_clf, X_train, y_train, X_test, 7)
ada_train, ada_test = get_stacking_base_datasets(ada_clf, X_train, y_train, X_test, 7)

In [ ]:
# 모델별 결과를 열 방향으로 합쳐 메타 모델의 최종 학습/테스트 데이터 생성
Stack_final_X_train = np.concatenate((knn_train, rf_train, dt_train, ada_train), axis=1)
Stack_final_X_test = np.concatenate((knn_test, rf_test, dt_test, ada_test), axis=1)

print('원본 학습 피처 데이터 Shape:', X_train.shape, '원본 테스트 피처 Shape:', X_test.shape)
print('스태킹 학습 피처 데이터 Shape:', Stack_final_X_train.shape,
      '스태킹 테스트 피처 데이터 Shape:', Stack_final_X_test.shape)

In [ ]:
lr_final.fit(Stack_final_X_train, y_train)
stack_final = lr_final.predict(Stack_final_X_test)

print('최종 메타 모델의 예측 정확도: {0:.4f}'.format(accuracy_score(y_test, stack_final)))

- 실제로는 개별 모델의 하이퍼 파라미터를 먼저 최적으로 튜닝한 뒤 스태킹을 구성한다. 스태킹 모델의 튜닝은 곧 개별 모델의 튜닝을 뜻한다.
- 스태킹은 분류뿐 아니라 회귀에도 적용할 수 있다.

## 정리

| 주제 | 핵심 |
|---|---|
| LightGBM | Leaf Wise 분할로 XGBoost와 비슷한 성능을 더 빠르게 낸다. 작은 데이터에서는 과적합에 주의한다 |
| LightGBM 튜닝 | `num_leaves`를 중심으로 `min_child_samples`, `max_depth`를 함께 조정한다 |
| HyperOpt | 베이지안 최적화로 Grid Search보다 적은 시도로 좋은 파라미터를 찾는다. 정수 변환과 -1 곱하기에 주의한다 |
| 스태킹 | 개별 모델의 예측값을 메타 모델이 다시 학습한다. CV 기반으로 만들면 과적합을 줄일 수 있다 |